In [1]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
# from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
from shapely.geometry import shape 
import json 
from shapely import wkt 
from shapely.geometry import Point
from shapely.geometry import box
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import matplotlib
from statsmodels.tsa.seasonal import seasonal_decompose

import glob
import os
import csv
import ast

from scipy.stats import chi2_contingency
from math import sqrt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import statsmodels.api as sm
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from pathlib import Path

In [2]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
DATA_DIR  = Path('../files')
PLOTS_DIR = Path('../outputs/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
(PLOTS_DIR / 'supplementary').mkdir(exist_ok=True)
(PLOTS_DIR / 'spatial').mkdir(exist_ok=True)

### Reading Temp & Precip (6km) 

In [4]:
temp_all = pd.read_csv(DATA_DIR / 'processed_weather_data/temp_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])
temp_all['time'] = pd.to_datetime(temp_all['time'])

precip_all = pd.read_csv(DATA_DIR / 'processed_weather_data/precip_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])
precip_all['time'] = pd.to_datetime(precip_all['time'])

### Wind (6km) 

In [5]:
hourly_wind_df = pd.read_csv(DATA_DIR / 'processed_weather_data/ea_hourly_wind_avg_6km.csv').drop(columns=['Unnamed: 0'])
hourly_wind_df['Timestamp'] = pd.to_datetime(hourly_wind_df['Timestamp'])
hourly_wind_df = hourly_wind_df.rename(columns = {'Timestamp':'time'})
hourly_wind_df['Date'] = hourly_wind_df['time'].dt.date

### Lightning (6km) 

In [6]:
hourly_lightning_df = pd.read_csv(DATA_DIR / 'processed_weather_data/ea_hourly_lightning_avg_6km_buffer_aligned.csv').drop(columns=['Unnamed: 0'])
hourly_lightning_df['Timestamp'] = pd.to_datetime(hourly_lightning_df['Timestamp'])
hourly_lightning_df = hourly_lightning_df.rename(columns = {'Timestamp':'time'})
hourly_lightning_df['Date'] = hourly_lightning_df['time'].dt.date

### Extreme Definitions 

### Temp 

In [7]:
# temp_all['Temp'].quantile(0.95)

In [8]:
temp_all['Date'] = temp_all['time'].dt.floor('D')

# Hot hour flag
temp_all['Hot_Hour'] = temp_all['Temp'] > 32

# Group by EA and Date, and count Hot_Hour sum
daily_hot_hours = (
    temp_all.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Hot_Hour': 'sum'})
)

daily_hot_hours = daily_hot_hours.rename(columns={'Hot_Hour': 'Num_Hot_Hours'})

### Precip 

In [9]:
precip_all['Date'] = precip_all['time'].dt.floor('D')  

# Group by EA and Date, sum Precip
daily_precip = (
    precip_all.groupby(['ea_code9ch', 'Date'], as_index=False)
      .agg({'Precip': 'sum'})
)

### Wind 

In [10]:
# Windy hour flag
hourly_wind_df['Windy_Hour'] = hourly_wind_df['Wind Gusts (m/s)'] > 5.93

# Group by EA and Date, and count Windy_Hour sum
daily_windy_hours = (
    hourly_wind_df.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Windy_Hour': 'sum'})
)

daily_windy_hours = daily_windy_hours.rename(columns={'Windy_Hour': 'Num_Windy_Hours'})

### Lightning 

In [11]:
daily_lightning_per_ea = hourly_lightning_df.groupby(['ea_code9ch', 'Date'])['Lightning Events'].sum().reset_index()

## Percentiles 

In [12]:
### 90th pct 
hot_hrs_90_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.90)
temp_90_thresh = temp_all['Temp'].quantile(0.90)
precip_90_thresh = daily_precip['Precip'].quantile(0.90)
lightning_90_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.90)
windy_hrs_90_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.90)
wind_90_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.90)


### 95th pct 
hot_hrs_95_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.95)
temp_95_thresh = temp_all['Temp'].quantile(0.95)
precip_95_thresh = daily_precip['Precip'].quantile(0.95)
lightning_95_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.95)
windy_hrs_95_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.95)
wind_95_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.95)


### 99th pct 
hot_hrs_99_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.99)
temp_99_thresh = temp_all['Temp'].quantile(0.99)
precip_99_thresh = daily_precip['Precip'].quantile(0.99)
lightning_99_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.99)
windy_hrs_99_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.99)
wind_99_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.99)

#### Dictionary for percentiles 

In [13]:
# Percentile Threshold dictionaries 
temp_thresh_dict = {
    '90': temp_90_thresh.round(2),
    '95': temp_95_thresh.round(2),
    '99': temp_99_thresh.round(2)
}

hot_hrs_thresh_dict = {
    '90': hot_hrs_90_thresh.round(2),
    '95': hot_hrs_95_thresh.round(2),
    '99': hot_hrs_99_thresh.round(2)
}

precip_thresh_dict = {
    '90': precip_90_thresh.round(2),
    '95': precip_95_thresh.round(2),
    '99': precip_99_thresh.round(2)
}

wind_thresh_dict = {
    '90': wind_90_thresh.round(2),
    '95': wind_95_thresh.round(2),
    '99': wind_99_thresh.round(2)
}

windy_hrs_thresh_dict = {
    '90': windy_hrs_90_thresh.round(2),
    '95': windy_hrs_95_thresh.round(2),
    '99': windy_hrs_99_thresh.round(2)
}

lightning_thresh_dict = {
    '90': 1,   # at least 1 lightning strike 
    '95': lightning_95_thresh.round(2),
    '99': lightning_99_thresh.round(2)
}

### Read EAs and CESI level 

In [14]:
eas_n_cesi_level = pd.read_csv(DATA_DIR / 'miscellaneous/eas_cesi_level_214_eas.csv')

### EA Names mapped  

In [15]:
ea_names_mapped = pd.read_csv(DATA_DIR / 'miscellaneous/ea_names_mapped.csv')

ea_names_mapped = ea_names_mapped[['ea_code9ch', 'LOC_NAME', 'BASE_NAM', 'geometry']]

### EAs n Sites  (6km) 

In [16]:
merged_eas_sites = pd.read_csv(DATA_DIR / 'miscellaneous/ea_site_list_6km_buffer.csv')
merged_eas_sites = merged_eas_sites[['ea_code9ch', 'Intersecting_Sites']]

# Convert the string representation of lists to actual lists
merged_eas_sites['Intersecting_Sites'] = merged_eas_sites['Intersecting_Sites'].apply(ast.literal_eval)

In [17]:
# merged_eas_sites['ea_code9ch'].nunique()

In [18]:
filtered_eas_sites_copy = merged_eas_sites

In [19]:
# EA -> 30200014, 30500020 

In [20]:
def prepare_hourly_weather_df_TPLW(
    ea_row, 
    temp_df, 
    precip_df, 
    lightning_df, 
    wind_df, 
):
    ea = ea_row['ea_code9ch']
    site_list = [int(s) for s in ea_row['Intersecting_Sites']]
    all_sites_dfs = []

    for site_id in site_list:
        # --- Temperature ---
        temp_filt = (
            temp_df[temp_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            .copy()
        )
        if 'Hot_Hour' in temp_filt.columns:
            temp_filt = temp_filt.drop(columns=['Hot_Hour'])
        temp_filt['time'] = pd.to_datetime(temp_filt['time'])

        # --- Precipitation ---
        precip_filt = (
            precip_df[precip_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Precip']]
        )
        precip_filt['time'] = pd.to_datetime(precip_filt['time'])

        # Merge temp + precip
        merged = temp_filt.merge(precip_filt, on='time', how='outer')

        # --- Lightning ---
        lightning_filt = (
            lightning_df[lightning_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Lightning Events']]
        )
        merged = merged.merge(lightning_filt, on='time', how='outer')
        merged['Lightning Events'] = merged['Lightning Events'].fillna(0)

        # --- Wind ---
        wind_filt = (
            wind_df[wind_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Wind Gusts (m/s)']]
        )
        merged = merged.merge(wind_filt, on='time', how='inner')

        # Add site_id
        merged['site_id'] = site_id

        all_sites_dfs.append(merged)

    # Concatenate all sites
    df_all = pd.concat(all_sites_dfs, ignore_index=True)

    # Aggregate by timestamp 
    agg_dict = {
        'Temp': 'mean',  # average across sites
        'Precip': 'mean',
        'Lightning Events': 'mean',
        'Wind Gusts (m/s)': 'mean',
        'site_id': lambda x: ','.join(map(str, x))  # list all contributing sites
    }

    df_agg = df_all.groupby('time', as_index=False).agg(agg_dict)

    # Add EA code
    df_agg['ea_code9ch'] = ea

    # Optional: reorder columns
    df_agg = df_agg[
        ['time', 'ea_code9ch', 'site_id', 'Temp', 'Precip', 
         'Lightning Events', 'Wind Gusts (m/s)']
    ]

    return df_agg

## Remove sites with > `10%` missing data & less than 24 months 

In [21]:
sites_to_omit = pd.read_csv(DATA_DIR / 'miscellaneous/complete_site_removal_df.csv')['site_id'].to_list()

### Resample hourly weather data -> Daily 

In [22]:
filtered_eas_sites_copy_r1 = merged_eas_sites.copy()

In [23]:
# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_weather_df_TPLW(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

In [24]:
merged_hourly_data_global = merged_hourly_data_global.loc[
    ~merged_hourly_data_global['site_id'].isin(sites_to_omit)
]

In [25]:
### Filter 214 EAs 

tplw_eas = pd.read_csv(DATA_DIR / 'miscellaneous/unique_ea_codes_TPLW.csv')

list_EAs = tplw_eas['ea_code9ch'].unique().tolist()

hourly_global_test = merged_hourly_data_global[merged_hourly_data_global['ea_code9ch'].isin(list_EAs)].reset_index(drop=True)

In [26]:
hourly_global_test['ea_code9ch'].nunique()

214

In [27]:
# 1. Convert to string (just in case)
hourly_global_test['site_id'] = hourly_global_test['site_id'].astype(str)

# 2. Split into list
hourly_global_test['site_id'] = hourly_global_test['site_id'].str.split(',')

# 3. Explode into separate rows
hourly_global_test = hourly_global_test.explode('site_id')

# 4. Clean up (remove spaces + convert to int)
hourly_global_test['site_id'] = hourly_global_test['site_id'].str.strip().astype(int)

### Filter Weather data 

In [28]:
hourly_temperature = hourly_global_test.drop(columns = ['Precip', 'Lightning Events', 'Wind Gusts (m/s)']).reset_index(drop=True)

# *** Regression Analysis 

### read hourly voltage, EA <-> CESI Mapping & Merge  

### read for all sensors 

In [29]:
hourly_voltage_22 = pd.read_csv(DATA_DIR / 'processed_undervolt_data/undervolt_hourly_voltages_df_22.csv')
hourly_voltage_23 = pd.read_csv(DATA_DIR / 'processed_undervolt_data/undervolt_hourly_voltages_df_23.csv')

hourly_voltage = pd.concat([hourly_voltage_22, hourly_voltage_23], axis=0, ignore_index=True)
hourly_voltage['time'] = pd.to_datetime(hourly_voltage['time'])

### sensors of interest 

In [30]:
sensor_of_interest_list = pd.read_csv(DATA_DIR / 'miscellaneous/common_sensors_22_23_rev4.csv')['respondent_id'].unique().tolist()

hourly_voltage = hourly_voltage[hourly_voltage['sensor_id'].isin(sensor_of_interest_list)].reset_index(drop=True)

In [31]:
## EA <-> CESI Mapping 
sensor_ea_mapping = pd.read_csv(DATA_DIR / 'miscellaneous/sensor_ea_mapping_rev1.csv').drop(columns=['Unnamed: 0'])

## Merge 
hourly_voltage = hourly_voltage.merge(sensor_ea_mapping, on = ['sensor_id', 'site_id'], how = 'left')

### Combine weather & voltage 

In [32]:
hourly_volt_n_temp = hourly_voltage.merge(hourly_temperature, on = ['time', 'site_id', 'ea_code9ch'], how = 'left')
hourly_volt_n_temp = hourly_volt_n_temp[['time', 'ea_code9ch', 'site_id', 'sensor_id', 'voltage', 'Temp', 'cesi_level']]
hourly_volt_n_temp['Temp'] = hourly_volt_n_temp['Temp'].round(3)

### Remove NaNs 

In [33]:
hourly_volt_n_temp = hourly_volt_n_temp.dropna().reset_index(drop=True)

### flag undervolts and extreme temps 

In [34]:
def add_flags(df, volt_thresh, temp_thresh):
    df = df.copy()

    # Undervoltage flag
    df['undervolt'] = (df['voltage'] < volt_thresh).astype(int)

    # Extreme temperature flag
    df['extreme_temp'] = (df['Temp'] >= temp_thresh).astype(int)

    return df

### specify hour undervolt threshold 

In [35]:
hr_undervolt_thresh = 207 

### add `undervolt` & `extreme_temp` flags 

In [36]:
hourly_volt_n_temp_flagged = add_flags(
        hourly_volt_n_temp, 
        volt_thresh = hr_undervolt_thresh,        ## undervolt hour thresh 
        temp_thresh = temp_thresh_dict['95'],     ## 95th percentile temp 
)

# re-order 
hourly_volt_n_temp_flagged = hourly_volt_n_temp_flagged[['time', 'ea_code9ch', 'site_id', 'sensor_id', 
                                                         'voltage', 'undervolt', 
                                                         'Temp', 'extreme_temp', 
                                                        'cesi_level']]

# A. Pooled Results (*all sensors together*)

### Same hour interaction (undervolt & high temp) 

In [37]:
def run_pooled_undervolt_logit(
    df,
    outcome_col='undervolt',
    exposure_col='extreme_temp',
    time_col='time',
    cluster_col='sensor_id',
    add_month_fe=True,
    add_hour_fe=True,
    use_cluster=True,
    disp=0
):

    df = df.copy()

    # Ensure datetime
    df[time_col] = pd.to_datetime(df[time_col])

    # Time features
    df['month'] = df[time_col].dt.month
    df['hour'] = df[time_col].dt.hour

    # Ensure binary
    df[outcome_col] = df[outcome_col].astype(int)
    df[exposure_col] = df[exposure_col].astype(int)

    # Outcome
    y = df[outcome_col]

    # Predictors
    X = df[[exposure_col]].copy()

    if add_month_fe:
        X['month'] = df['month']

    if add_hour_fe:
        X['hour'] = df['hour']

    # Fixed effects
    fe_cols = []
    if add_month_fe:
        fe_cols.append('month')

    if add_hour_fe:
        fe_cols.append('hour')

    if fe_cols:
        X = pd.get_dummies(X, columns=fe_cols, drop_first=True)

    X = X.astype(float)
    X = sm.add_constant(X)

    # Fit model
    if use_cluster:
        model = sm.Logit(y, X).fit(
            cov_type='cluster',
            cov_kwds={'groups': df[cluster_col]},
            disp=disp
        )
    else:
        model = sm.Logit(y, X).fit(disp=disp)

    return model

## *** 1) Define Heat Spells 

## *** Undervoltages unlike outages (daily) are at the hourly resolution 

### so to define a heat spell, we look at what is extreme for diff hour windows (4, 6, 8, 12, 24, 48) 

### use `discrete blocks` 

### calculate the 95th percentile of number of high temp hours within the block 

In [38]:
import pandas as pd
import numpy as np

def define_heat_spell_from_time_blocks(
    df,
    temp_col='Temp',
    time_col='time',
    ea_col='ea_code9ch',
    sensor_col='sensor_id',
    threshold_temp=32.16,
    block_hours=4,
    percentile=95,
    block_origin='start_day'
):
    """
    Defines heat spell flags using discrete, non-overlapping time blocks.

    Steps:
    1. Aggregates temperature to one value per EA-hour.
    2. Flags hours where temperature >= threshold_temp.
    3. Assigns each hour to a fixed time block.
    4. Counts extreme-temp hours within each EA-block.
    5. Finds the percentile threshold of block counts.
    6. Flags heat spell blocks where count >= percentile threshold.
    """

    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])

    # --------------------------------------------------
    # 1. Collapse to one temperature value per EA-hour
    # --------------------------------------------------
    ea_hourly = (
        df.groupby([ea_col, time_col], as_index=False)
          .agg(
              Temp_mean=(temp_col, 'mean'),
              n_sensors=(sensor_col, 'nunique')
          )
    )

    # --------------------------------------------------
    # 2. Flag extreme temperature hours
    # --------------------------------------------------
    ea_hourly['extreme_temp_hour'] = (
        ea_hourly['Temp_mean'] >= threshold_temp
    ).astype(int)

    # --------------------------------------------------
    # 3. Assign discrete non-overlapping time blocks
    # --------------------------------------------------
    freq = f'{block_hours}h'

    ea_hourly['time_block_start'] = (
        ea_hourly[time_col]
        .dt.floor(freq)
    )

    ea_hourly['time_block_end'] = (
        ea_hourly['time_block_start']
        + pd.to_timedelta(block_hours, unit='h')
    )

    # --------------------------------------------------
    # 4. Count extreme-temp hours within each EA-block
    # --------------------------------------------------
    block_df = (
        ea_hourly
        .groupby([ea_col, 'time_block_start', 'time_block_end'], as_index=False)
        .agg(
            extreme_hours_in_block=('extreme_temp_hour', 'sum'),
            total_hours_in_block=('extreme_temp_hour', 'count'),
            Temp_mean_block=('Temp_mean', 'mean'),
            Temp_max_block=('Temp_mean', 'max'),
            n_sensors_mean=('n_sensors', 'mean')
        )
    )

    # --------------------------------------------------
    # 5. Find percentile threshold
    # --------------------------------------------------
    heat_spell_threshold = np.nanpercentile(
        block_df['extreme_hours_in_block'],
        percentile
    )

    # Since block counts are discrete, round up
    heat_spell_threshold = int(np.ceil(heat_spell_threshold))

    # --------------------------------------------------
    # 6. Flag heat spell blocks
    # --------------------------------------------------
    block_df[f'heat_spell_{block_hours}hr'] = (
        block_df['extreme_hours_in_block'] >= heat_spell_threshold
    ).astype(int)

    # --------------------------------------------------
    # 7. Keep useful metadata
    # --------------------------------------------------
    block_df.attrs['threshold_temp'] = threshold_temp
    block_df.attrs['block_hours'] = block_hours
    block_df.attrs['percentile'] = percentile
    block_df.attrs['heat_spell_threshold_hours'] = heat_spell_threshold

    print(f"Extreme temp threshold: Temp >= {threshold_temp}°C")
    print(f"Discrete block size: {block_hours} hours")
    print(f"{percentile}th percentile block-count threshold: {heat_spell_threshold} extreme hours")
    print(
        f"Heat spell definition: at least {heat_spell_threshold} "
        f"extreme-temp hours within a discrete {block_hours}-hour block"
    )

    return block_df

In [39]:
heat_spell_block_df = define_heat_spell_from_time_blocks(
    hourly_volt_n_temp,
    threshold_temp = temp_thresh_dict['95'],
    block_hours = 4,
    percentile = 95
)

Extreme temp threshold: Temp >= 32.16°C
Discrete block size: 4 hours
95th percentile block-count threshold: 2 extreme hours
Heat spell definition: at least 2 extreme-temp hours within a discrete 4-hour block


In [40]:
heat_spell_block_df

,ea_code9ch,time_block_start,time_block_end,extreme_hours_in_block,total_hours_in_block,Temp_mean_block,Temp_max_block,n_sensors_mean,heat_spell_4hr
0,30200002,2022-01-01 00:00:00,2022-01-01 04:00:00,0,4,25.87700,26.076,2.0,0
1,30200002,2022-01-01 04:00:00,2022-01-01 08:00:00,0,4,25.69075,26.686,2.0,0
2,30200002,2022-01-01 08:00:00,2022-01-01 12:00:00,0,4,30.31175,31.647,2.0,0
3,30200002,2022-01-01 12:00:00,2022-01-01 16:00:00,0,4,31.74550,32.109,2.0,0
4,30200002,2022-01-01 16:00:00,2022-01-01 20:00:00,0,4,29.47000,30.639,2.0,0
...,...,...,...,...,...,...,...,...,...
830686,31300407,2023-12-31 04:00:00,2023-12-31 08:00:00,0,4,26.27100,26.837,3.0,0
830687,31300407,2023-12-31 08:00:00,2023-12-31 12:00:00,1,4,30.69400,32.870,3.0,0
830688,31300407,2023-12-31 12:00:00,2023-12-31 16:00:00,3,4,32.95850,33.687,3.0,1
830689,31300407,2023-12-31 16:00:00,2023-12-31 20:00:00,0,4,29.62400,31.016,3.0,0


### map heat spells back to hourly volt & temp df 

### all hours with a heat spell `flagged` as 1 

In [41]:
def map_heat_spell_blocks_to_hourly(
    hourly_df,
    heat_spell_block_df,
    time_col='time',
    ea_col='ea_code9ch',
    sensor_col='sensor_id',
    block_start_col='time_block_start',
    block_end_col='time_block_end',
    heat_spell_col=None
):
    """
    Maps EA-level heat spell block flags back onto sensor-hour rows.
    Automatically detects the heat spell column if heat_spell_col is not passed.
    """

    hourly_df = hourly_df.copy()
    block_df = heat_spell_block_df.copy()

    hourly_df[time_col] = pd.to_datetime(hourly_df[time_col])
    block_df[block_start_col] = pd.to_datetime(block_df[block_start_col])
    block_df[block_end_col] = pd.to_datetime(block_df[block_end_col])

    # --------------------------------------------------
    # Automatically detect heat spell column
    # --------------------------------------------------
    if heat_spell_col is None:
        heat_spell_cols = [
            col for col in block_df.columns
            if col.startswith('heat_spell')
        ]

        if len(heat_spell_cols) == 0:
            raise ValueError("No heat spell column found. Expected a column starting with 'heat_spell'.")

        if len(heat_spell_cols) > 1:
            raise ValueError(
                f"Multiple heat spell columns found: {heat_spell_cols}. "
                "Please pass heat_spell_col explicitly."
            )

        heat_spell_col = heat_spell_cols[0]

    # Keep only needed block-level columns
    block_cols = [
        ea_col,
        block_start_col,
        block_end_col,
        heat_spell_col
    ]

    block_df = block_df[block_cols]

    # Infer block size from block start/end
    block_hours = int(
        (block_df[block_end_col].iloc[0] - block_df[block_start_col].iloc[0])
        / pd.Timedelta(hours=1)
    )

    # Assign each hourly row to its matching block start
    hourly_df[block_start_col] = hourly_df[time_col].dt.floor(f'{block_hours}h')

    # Merge block-level heat spell flag onto hourly sensor data
    merged = hourly_df.merge(
        block_df.drop(columns=[block_end_col]),
        on=[ea_col, block_start_col],
        how='left'
    )

    # Fill missing heat spell flags with 0
    merged[heat_spell_col] = merged[heat_spell_col].fillna(0).astype(int)

    # Sort final output
    merged = merged.sort_values(
        [ea_col, sensor_col, time_col]
    ).reset_index(drop=True)

    return merged

In [42]:
hourly_volt_n_temp_heat_spell = map_heat_spell_blocks_to_hourly(
    hourly_volt_n_temp_flagged,
    heat_spell_block_df,
)

## *** 2) Define Sustained Undervoltages 

In [43]:
def add_consecutive_undervolt_flag(
    df,
    time_col='time',
    sensor_col='sensor_id',
    undervolt_col='undervolt',
    min_hours=2
):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])

    # Sort so consecutive logic works
    df = df.sort_values([sensor_col, time_col])

    # New event starts when undervolt status changes
    df['event_group'] = (
        df.groupby(sensor_col)[undervolt_col]
          .transform(lambda x: x.ne(x.shift()).cumsum())
    )

    # Count consecutive rows within each undervolt/non-undervolt spell
    df['spell_length'] = (
        df.groupby([sensor_col, 'event_group'])[undervolt_col]
          .transform('size')
    )

    # Flag only undervolt rows that belong to spells lasting >= min_hours
    df[f'undervolt_{min_hours}hr_plus'] = (
        (df[undervolt_col] == 1) &
        (df['spell_length'] >= min_hours)
    ).astype(int)

    # Drop helper columns
    df = df.drop(columns=['event_group', 'spell_length'])

    # 🔥 Sort by site_id (and time for sanity), then reset index
    df = df.sort_values(['site_id', time_col]).reset_index(drop=True)

    return df

### Specify the undervoltage duration (1 hour) 

In [44]:
unv_flagged_1hr = add_consecutive_undervolt_flag(
    hourly_volt_n_temp_flagged,
    min_hours = 1
)

## *** 3) Undervoltages & Heat Spell 

### specify the dfs & merge 

In [45]:
df_sustained_undervolt = unv_flagged_1hr.copy()
df_sustained_temp = hourly_volt_n_temp_heat_spell.copy()

In [46]:
# Dynamically detect columns
heat_col = [c for c in df_sustained_temp.columns if 'heat_spell' in c][0]
undervolt_col = [c for c in df_sustained_undervolt.columns if 'undervolt_' in c and 'hr_plus' in c][0]

merge_cols = ['time', 'ea_code9ch', 'site_id', 'sensor_id', 'voltage', 'undervolt', 'Temp', 'extreme_temp', 'cesi_level']

# Merge
hourly_combined_heat_n_temp = df_sustained_undervolt.merge(
    df_sustained_temp[merge_cols + [heat_col]],
    on=merge_cols,
    how='left'
)

hourly_combined_heat_n_temp = hourly_combined_heat_n_temp.sort_values(['site_id', 'sensor_id', 'time']).reset_index(drop=True)

print(f"Using outcome column: {undervolt_col}")
print(f"Using exposure column: {heat_col}")

Using outcome column: undervolt_1hr_plus
Using exposure column: heat_spell_4hr


### combined df having heat spell & specified undervolt duration  

In [47]:
hourly_combined_heat_n_temp.head()

,time,ea_code9ch,site_id,sensor_id,voltage,undervolt,Temp,extreme_temp,cesi_level,undervolt_1hr_plus,heat_spell_4hr
0,2022-01-01 00:00:00,30410207,1,bb3ff846,236.64,0,26.221,0,Low,0,0
1,2022-01-01 01:00:00,30410207,1,bb3ff846,235.38,0,26.100,0,Low,0,0
2,2022-01-01 02:00:00,30410207,1,bb3ff846,235.45,0,25.953,0,Low,0,0
3,2022-01-01 03:00:00,30410207,1,bb3ff846,236.54,0,25.780,0,Low,0,0
4,2022-01-01 04:00:00,30410207,1,bb3ff846,237.72,0,25.564,0,Low,0,0


### Run the model 

In [48]:
# Run model
model = run_pooled_undervolt_logit(
    hourly_combined_heat_n_temp,
    outcome_col = undervolt_col,
    exposure_col = heat_col,
    cluster_col = 'sensor_id'
)

print(model.summary())

                           Logit Regression Results                           
Dep. Variable:     undervolt_1hr_plus   No. Observations:              6135649
Model:                          Logit   Df Residuals:                  6135613
Method:                           MLE   Df Model:                           35
Date:                Sun, 21 Jun 2026   Pseudo R-squ.:                 0.04302
Time:                        05:13:22   Log-Likelihood:            -1.3625e+06
converged:                       True   LL-Null:                   -1.4237e+06
Covariance Type:              cluster   LLR p-value:                     0.000
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -2.8081      0.149    -18.790      0.000      -3.101      -2.515
heat_spell_4hr     0.1190      0.028      4.270      0.000       0.064       0.174
month_2           -0.0041      0.046

## Pooled model with CESI interaction 

### Same hour interaction (undervolt & heat spell...`with CESI`) 

In [49]:
def run_pooled_undervolt_logit_cesi(
    df,
    outcome_col='undervolt',
    exposure_col='extreme_temp',
    time_col='time',
    cluster_col='sensor_id',
    add_month_fe=True,
    add_hour_fe=True,
    use_cluster=True,
    disp=0,
    cesi_col='cesi_level',
    cesi_ref='Low',
    cesi_interact_cols=None
):
    df = df.copy()
    # Ensure datetime
    df[time_col] = pd.to_datetime(df[time_col])
    # Time features
    df['month'] = df[time_col].dt.month
    df['hour'] = df[time_col].dt.hour
    # Ensure binary
    df[outcome_col] = df[outcome_col].astype(int)
    df[exposure_col] = df[exposure_col].astype(int)
    # Outcome
    y = df[outcome_col]
    # Predictors
    X = df[[exposure_col]].copy()

    # CESI dummy + interactions
    if cesi_col is not None:
        df['cesi_High'] = (df[cesi_col] != cesi_ref).astype(int)
        X['cesi_High'] = df['cesi_High']

        # Default: interact exposure_col with cesi_High if no explicit list given
        if cesi_interact_cols is None:
            cesi_interact_cols = [exposure_col]

        for col in cesi_interact_cols:
            int_col = f'{col}_x_High'
            X[int_col] = df[col] * df['cesi_High']

    if add_month_fe:
        X['month'] = df['month']
    if add_hour_fe:
        X['hour'] = df['hour']
    # Fixed effects
    fe_cols = []
    if add_month_fe:
        fe_cols.append('month')
    if add_hour_fe:
        fe_cols.append('hour')
    if fe_cols:
        X = pd.get_dummies(X, columns=fe_cols, drop_first=True)
    X = X.astype(float)
    X = sm.add_constant(X)
    # Fit model
    if use_cluster:
        model = sm.Logit(y, X).fit(
            cov_type='cluster',
            cov_kwds={'groups': df[cluster_col]},
            disp=disp
        )
    else:
        model = sm.Logit(y, X).fit(disp=disp)
    return model

In [50]:
# Run model
model = run_pooled_undervolt_logit_cesi(
    hourly_combined_heat_n_temp,
    outcome_col = undervolt_col,
    exposure_col = heat_col,
    cluster_col = 'sensor_id'
)

print(model.summary())

                           Logit Regression Results                           
Dep. Variable:     undervolt_1hr_plus   No. Observations:              6135649
Model:                          Logit   Df Residuals:                  6135611
Method:                           MLE   Df Model:                           37
Date:                Sun, 21 Jun 2026   Pseudo R-squ.:                 0.06982
Time:                        05:14:30   Log-Likelihood:            -1.3243e+06
converged:                       True   LL-Null:                   -1.4237e+06
Covariance Type:              cluster   LLR p-value:                     0.000
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -3.1146      0.177    -17.582      0.000      -3.462      -2.767
heat_spell_4hr            0.1149      0.052      2.217      0.027       0.013       0.217
cesi_Hig